-----------
# **Midterm Project** – Data Cleaning & Preparation for Machine Learning

# **Dataset**: CTU-IoT-Malware-Capture-8-1conn

# **Objectives**: Prepare raw network traffic data for ML by performing EDA, cleaning, feature engineering, encoding, scaling, and export.
------

In [48]:
# =========================
# 1. IMPORT LIBRARIES
# =========================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# For plots
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['font.family'] = 'sans-serif'



In [49]:
# =========================
# 2. LOAD DATASET
# =========================
df = pd.read_csv("/content/CTU-IoT-Malware-Capture-8-1conn_Use_this_one-1 (1).csv", sep=None, engine="python")
df.columns = df.columns.str.strip()

# Quick look
print("RAW DATA SAMPLE:")
display(df.head())

print("\nDataset shape:", df.shape)
print("\nMissing values per column:\n", df.isnull().sum())


RAW DATA SAMPLE:


,ts,uid,id.orig_h,id.orig_p,id.resp_h,id.resp_p,proto,service,duration,orig_bytes,...,local_resp,missed_bytes,history,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,label,detailed-label
0,1.533043e+09,C5JLGOoxIw2dBZt47,192.168.100.113,123,81.2.254.224,123,udp,-,0.005490,48,...,-,0,Dd,1,76,1,76,-,Benign,-
1,1.533043e+09,Cf3cHf4jZr9nvD808i,192.168.100.113,123,147.231.100.5,123,udp,-,0.001741,48,...,-,0,Dd,1,76,1,76,-,Benign,-
2,1.533043e+09,CJgmSt3bSY6XwE9fzc,192.168.100.113,123,31.31.74.35,123,udp,-,0.004495,48,...,-,0,Dd,1,76,1,76,-,Benign,-
3,1.533043e+09,Cav32m4csR3OZYhShj,192.168.100.113,123,147.251.48.140,123,udp,-,0.006988,48,...,-,0,Dd,1,76,1,76,-,Benign,-
4,1.533043e+09,ClwPfA40tU9UT4nksg,192.168.100.113,123,147.231.100.5,123,udp,-,0.001487,48,...,-,0,Dd,1,76,1,76,-,Benign,-



Dataset shape: (10403, 23)

Missing values per column:
 ts                0
uid               0
id.orig_h         0
id.orig_p         0
id.resp_h         0
id.resp_p         0
proto             0
service           0
duration          0
orig_bytes        0
resp_bytes        0
conn_state        0
local_orig        0
local_resp        0
missed_bytes      0
history           0
orig_pkts         0
orig_ip_bytes     0
resp_pkts         0
resp_ip_bytes     0
tunnel_parents    0
label             0
detailed-label    0
dtype: int64


In [50]:
# =========================
# 3. INITIAL CLEANING
# =========================

# Replace "-" with NaN
df.replace("-", np.nan, inplace=True)

# Drop unneeded identifier columns
df.drop(columns=["uid", "id.orig_h", "id.resp_h", "id.resp_p", "history"], errors="ignore", inplace=True)


/tmp/ipykernel_447/4278317664.py:6: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df.replace("-", np.nan, inplace=True)


In [51]:
# =========================
# 4. TRAIN/TEST SPLIT
# =========================
train_df, test_df = train_test_split(df, test_size=0.30, random_state=42)

print("Training size:", train_df.shape)
print("Testing size:", test_df.shape)


Training size: (7282, 18)
Testing size: (3121, 18)


In [52]:
# =========================
# 5. CONVERT NUMERIC COLUMNS
# =========================
cols_to_fix = ["duration", "orig_bytes", "orig_pkts"]

for col in cols_to_fix:
   train_df[col] = pd.to_numeric(train_df[col], errors="coerce")
   test_df[col] = pd.to_numeric(test_df[col], errors="coerce")

# Fill missing values with median from training set
train_medians = train_df[cols_to_fix].median()
for col in cols_to_fix:
   train_df[col] = train_df[col].fillna(train_medians[col])
   test_df[col] = test_df[col].fillna(train_medians[col])


In [53]:
# =========================
# 6. FEATURE ENGINEERING
# =========================
train_df["bytes_per_packet"] = train_df["orig_bytes"] / (train_df["orig_pkts"] + 1)
test_df["bytes_per_packet"] = test_df["orig_bytes"] / (test_df["orig_pkts"] + 1)

train_df["packet_density"] = train_df["orig_pkts"] / (train_df["duration"] + 1)
test_df["packet_density"] = test_df["orig_pkts"] / (test_df["duration"] + 1)

print("Feature Engineering Check:")
print(train_df[["bytes_per_packet", "packet_density"]].head())

Feature Engineering Check:
      bytes_per_packet  packet_density
2089              24.0        0.995278
702               24.0        0.993061
503               24.0        0.993061
8847              24.0        0.993061
9172              24.0        0.993061


In [54]:
# =========================
# 7. ONE-HOT ENCODING CATEGORICAL FEATURES
# =========================
# Convert categorical columns to dummies
train_df = pd.get_dummies(train_df, drop_first=True)
test_df = pd.get_dummies(test_df, drop_first=True)

# Align test columns to training columns
test_df = test_df.reindex(columns=train_df.columns, fill_value=0)

print("\nEncoding Check:")
print("Total columns after encoding:", len(train_df.columns))
print([col for col in train_df.columns if "proto" in col or "conn_state" in col][:5])

print("\nAlignment Check:")
print("Train/Test same columns:", list(train_df.columns) == list(test_df.columns))


Encoding Check:
Total columns after encoding: 20
['proto_udp', 'conn_state_S0', 'conn_state_SF']

Alignment Check:
Train/Test same columns: True


In [55]:
# =========================
# 9. SPLIT FEATURES AND TARGET
# =========================
# Print columns to identify the correct target column name after one-hot encoding
print("Columns in train_df before splitting:")
print(train_df.columns)

# Based on the printed columns, the target column is 'label_Malicious'
X_train = train_df.drop("label_Malicious", axis=1)
y_train = train_df["label_Malicious"]

X_test = test_df.drop("label_Malicious", axis=1)
y_test = test_df["label_Malicious"]

# Verify shapes and label distribution
print("Training features shape:", X_train.shape)
print("Training target shape:", y_train.shape)
print("Test features shape:", X_test.shape)
print("Test target shape:", y_test.shape)

print("\nTraining label distribution:\n", y_train.value_counts())
print("\nTest label distribution:\n", y_test.value_counts())

Columns in train_df before splitting:
Index(['ts', 'id.orig_p', 'service', 'duration', 'orig_bytes', 'local_orig',
       'local_resp', 'missed_bytes', 'orig_pkts', 'orig_ip_bytes', 'resp_pkts',
       'resp_ip_bytes', 'tunnel_parents', 'bytes_per_packet', 'packet_density',
       'proto_udp', 'resp_bytes_48', 'conn_state_S0', 'conn_state_SF',
       'label_Malicious'],
      dtype='object')
Training features shape: (7282, 19)
Training target shape: (7282,)
Test features shape: (3121, 19)
Test target shape: (3121,)

Training label distribution:
 label_Malicious
True     5776
False    1506
Name: count, dtype: int64

Test label distribution:
 label_Malicious
True     2446
False     675
Name: count, dtype: int64


In [56]:
# =========================
# 10. SCALE FEATURES
# =========================
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert back to DataFrame (optional)
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test.columns)

# Quick sample
display(X_train_scaled.head())

print("\nScaling Check:")
print(X_train_scaled.head())
print("Mean (approx 0):", X_train_scaled.mean().mean())
print("Std (approx 1):", X_train_scaled.std().mean())

/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1101: RuntimeWarning: invalid value encountered in divide
  updated_mean = (last_sum + new_sum) / updated_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1106: RuntimeWarning: invalid value encountered in divide
  T = new_sum / new_sample_count
/usr/local/lib/python3.12/dist-packages/sklearn/utils/extmath.py:1126: RuntimeWarning: invalid value encountered in divide
  new_unnormalized_variance -= correction**2 / new_sample_count


,ts,id.orig_p,service,duration,orig_bytes,local_orig,local_resp,missed_bytes,orig_pkts,orig_ip_bytes,resp_pkts,resp_ip_bytes,tunnel_parents,bytes_per_packet,packet_density,proto_udp,resp_bytes_48,conn_state_S0,conn_state_SF
0,-1.042257,-1.898451,NaN,-0.501732,0.500772,NaN,NaN,0.0,-0.500236,-0.242308,1.967461,1.966897,NaN,0.499487,0.509071,1.960040,1.969119,-1.967461,1.969119
1,-1.506046,0.661165,NaN,-0.499934,0.500772,NaN,NaN,0.0,-0.500236,-0.583966,-0.508269,-0.508259,NaN,0.499487,0.488473,-0.510194,-0.507841,0.508269,-0.507841
2,-1.575895,0.115365,NaN,-0.499934,0.500772,NaN,NaN,0.0,-0.500236,-0.583966,-0.508269,-0.508259,NaN,0.499487,0.488473,-0.510194,-0.507841,0.508269,-0.507841
3,1.201202,0.851818,NaN,-0.499934,0.500772,NaN,NaN,0.0,-0.500236,-0.583966,-0.508269,-0.508259,NaN,0.499487,0.488473,-0.510194,-0.507841,0.508269,-0.507841
4,1.312409,0.318744,NaN,-0.499934,0.500772,NaN,NaN,0.0,-0.500236,-0.583966,-0.508269,-0.508259,NaN,0.499487,0.488473,-0.510194,-0.507841,0.508269,-0.507841



Scaling Check:
         ts  id.orig_p  service  duration  orig_bytes  local_orig  local_resp  \
0 -1.042257  -1.898451      NaN -0.501732    0.500772         NaN         NaN   
1 -1.506046   0.661165      NaN -0.499934    0.500772         NaN         NaN   
2 -1.575895   0.115365      NaN -0.499934    0.500772         NaN         NaN   
3  1.201202   0.851818      NaN -0.499934    0.500772         NaN         NaN   
4  1.312409   0.318744      NaN -0.499934    0.500772         NaN         NaN   

   missed_bytes  orig_pkts  orig_ip_bytes  resp_pkts  resp_ip_bytes  \
0           0.0  -0.500236      -0.242308   1.967461       1.966897   
1           0.0  -0.500236      -0.583966  -0.508269      -0.508259   
2           0.0  -0.500236      -0.583966  -0.508269      -0.508259   
3           0.0  -0.500236      -0.583966  -0.508269      -0.508259   
4           0.0  -0.500236      -0.583966  -0.508269      -0.508259   

   tunnel_parents  bytes_per_packet  packet_density  proto_udp  resp_b

In [57]:
# =========================
# 11. EXPORT CLEAN DATASETS
# =========================
train_df_final = pd.concat([X_train_scaled, y_train.reset_index(drop=True)], axis=1)
test_df_final = pd.concat([X_test_scaled, y_test.reset_index(drop=True)], axis=1)

train_df_final.to_csv("clean_training_dataset_scaled.csv", index=False)
test_df_final.to_csv("clean_testing_dataset_scaled.csv", index=False)


## Reflections on Data Preparation

During this project, several challenges were addressed to ensure a robust, ML-ready dataset:

**1. Avoiding Data Leakage**  
- Early EDA was initially performed before splitting the dataset, which risks data leakage.  
- All cleaning, missing value handling, and feature engineering were subsequently applied **using the training set only**, with test data processed based on training parameters.  

**2. Accidental Data Alteration**  
- Some initial commands unintentionally modified columns, particularly categorical ones.  
- Transformations were reordered and controlled to preserve original features and maintain reproducibility.  

**3. Median over Mean for Missing Values**  
- Numeric features such as `duration` and `orig_bytes` were filled with the **median** instead of the mean.  
- Network traffic data is often **skewed**: a few very long or large connections could pull the mean away from typical values.  
- Using the median ensures the imputed values reflect the central tendency of most connections.  

**4. Categorical Audit Prior to One-Hot Encoding**  
- Unique value counts were examined before encoding to verify the distribution of network protocols, services, and connection states.  
- This step ensured that rare categories were accounted for and that the binary sparse matrix used for ML models accurately reflected the data.  

**5. Lessons Learned**  
- Maintaining a clear pipeline (**clean → split → feature engineer → encode → scale**) prevents errors from propagating.  
- Documenting each preprocessing decision improves reproducibility and interpretability.  
- Visualizing categorical distributions before and after transformation helps confirm data integrity.

**6. Feature Engineering Rationale**  
- **Bytes per Packet:** Calculated as `orig_bytes / (orig_pkts + 1)` to measure the average payload size per packet, giving insight into traffic intensity per connection.  
- **Packet Density:** Calculated as `orig_pkts / (duration + 1)` to capture how rapidly packets are transmitted over time, highlighting unusual bursts typical of malicious activity.  
- These engineered features provide meaningful signals for ML models beyond raw counts and help differentiate benign from malicious network behavior.

## Best Practices Followed

- **Stepwise, reproducible pipeline:** Cleaning → Train/Test Split → Feature Engineering → Encoding → Scaling → Export, ensuring no accidental data leakage.  
- **Data integrity audits:** Checked for missing values, verified categorical distributions, and confirmed feature ranges before and after transformations.  
- **Robust statistical choices:** Median imputation for skewed numeric data, which prevents outliers from distorting the central tendency.  
- **Meaningful feature engineering:** Created features (bytes per packet, packet density) that provide additional insight for ML models.  
- **Documentation and transparency:** Each preprocessing decision is clearly explained to ensure reproducibility and interpretability, demonstrating good data science practice.